# 第57课 · 从一个神经元到一张网——MLP 从零搭建，41 个参数全由你亲手点亮

**目标**：实现 `Neuron → Layer → MLP`，每层支持 forward / backward。

> **分层叙事**：1 神经元 → 一层 → 整网；参数表跟着代码走。权重**不能全 0**（对称性陷阱）。

🔗 **Aurora 连接**：这个 MLP 和 PyTorch `nn.Module` 完全同构，也是 Aurora ML 引擎（`src/aurora/ml/`，计划中模块）的原型；**L61** `L61_pytorch_nn.ipynb` 将用 PyTorch 复现相同的三层结构，届时你可以逐行对照两份代码，验证自己的实现和框架行为一致。

← **上一课**　[L56 · 反向传播手推](L56_backward_pass.ipynb)

> 上节课学习了 **反向传播手推**：链式法则逐层展开，梯度 = 局部梯度 × 上游梯度。  
> 本课将探讨 **MLP 从零搭建**。

## 本课剧情：从单个神经元到"会说话"的网络

有没有想过，为什么 GPT 里几十亿个参数都可以用同一行代码 `loss.backward()` 更新？

答案藏在它的**组装方式**里。神经网络不是一个大黑箱，而是三层嵌套：

```
Neuron  ← 最小单元：一个加权求和 + 激活函数
Layer   ← 中间层：nout 个 Neuron 并行，接收同一份输入
MLP     ← 完整网络：多个 Layer 首尾相连
```

**一个 Neuron 做的事**（nin=2，输出一个标量）：
```
output = relu( w[0]*x[0] + w[1]*x[1] + b )
       = relu( Value(0.7)*Value(1.0) + Value(-0.3)*Value(2.0) + Value(0.1) )
       = relu( Value(0.7 - 0.6 + 0.1) )
       = relu( Value(0.2) ) = Value(0.2)
```

每一步都是上节课实现的 `Value` 算子，计算图自动建立，`backward()` 自动传梯度。

**为什么这样设计**：

| 层级 | 角色 | 参数数 |
|---|---|---|
| Neuron(nin) | 1 个线性加权 + 激活 | nin+1 |
| Layer(nin, nout) | nout 个 Neuron | nout×(nin+1) |
| MLP(nin, [n1,n2,...]) | 多层串联 | 逐层累加 |

本节任务：实现三个 class，让 `MLP(3,[4,4,1])` 能前向、能 `backward()`、能数参数。

In [ ]:
import random

# ── Value 类（来自 L54–L56 的完整实现，这里直接粘贴以便本课独立运行）──
import math

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other): return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other): return self * other

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) + (-self)

    def __truediv__(self, other):
        # 必须用 other**-1（调用 __pow__），而非 Value(other.data**-1)。
        # Value(x**-1) 创建孤立节点：不连接到原来的 other，梯度无法回流。
        # other**-1 以 other 为子节点构建 1/other 节点，梯度链路完整。
        other = other if isinstance(other, Value) else Value(other)
        return self * other**-1

    def __pow__(self, exp):
        assert isinstance(exp, (int, float))
        out = Value(self.data**exp, (self,), f'**{exp}')
        def _backward():
            self.grad += exp * (self.data**(exp-1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

# 快速冒烟测试
a = Value(2.0); b = Value(3.0)
c = (a * b + Value(1.0)).tanh()
c.backward()
assert abs(a.grad) > 0, "Value.backward() 未工作"
print("✅ Value 类加载成功")

# 除法梯度验证（节点连通性检查）
x = Value(6.0)
y = Value(2.0)
z = x / y          # z = 3.0
z.backward()
# d(x/y)/dx = 1/y = 0.5,  d(x/y)/dy = -x/y^2 = -1.5
assert abs(x.grad - 0.5) < 1e-9,   f"dz/dx 错误: {x.grad}（应为 0.5）"
assert abs(y.grad - (-1.5)) < 1e-9, f"dz/dy 错误: {y.grad}（应为 -1.5）"
print(f"✅ 除法梯度正确：dz/dx={x.grad:.4f}, dz/dy={y.grad:.4f}")

## 1. Neuron — 单个神经元

**实现要求**：

```python
class Neuron(nin, nonlin=True):
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(0.0)

    def __call__(x):  # x: list[Value or float]
        act = sum(wi * xi for wi, xi in zip(self.w, x)) + self.b
        return act.tanh() if self.nonlin else act

    def parameters() → list[Value]  # 返回 w + [b]
```

（micrograd 原版用 `relu`，本课统一用 `tanh` 作隐藏层激活——输出落在 (-1, 1)，便于观察；两者都是 L55 已实现的算子，可自行替换对比。）

**验收标准**：
- `Neuron(3)(x)` 返回单个 `Value`，且 `.data` 有限
- `len(Neuron(3).parameters()) == 4`（3 个 w + 1 个 b）
- `nonlin=False` 时直接返回线性组合，不过 tanh

### 参数数量计算：为什么 Layer(nin, nout) 有 nout×(nin+1) 个参数？

看起来只是一个公式，但理解背后的逻辑能帮你快速验证任何网络的参数总数。

**从一个 Neuron 开始**：
- `Neuron(nin)` 需要 `nin` 个权重（w[0], w[1], ..., w[nin-1]）加 1 个偏置（b）
- 总共 **nin + 1** 个参数

**升级到一个 Layer**：
- `Layer(nin, nout)` 包含 `nout` 个独立的 `Neuron(nin)`
- 第 1 个 Neuron 有 nin+1 个参数
- 第 2 个 Neuron 有 nin+1 个参数
- ...
- 第 nout 个 Neuron 有 nin+1 个参数

**加起来**：
```
Layer 的总参数 = nout 个 Neuron × 每个 (nin+1) 参数
              = nout × (nin + 1)
              = nout×nin + nout×1
              = (权重数) + (偏置数)
```

**具体例子**：`Layer(3, 4)` 有 4 个 Neuron，每个处理 3 个输入
- 权重数 = 3 × 4 = 12（每个 Neuron 3 个权重，4 个 Neuron）
- 偏置数 = 4（每个 Neuron 1 个偏置）
- **总计 = 16**

这就是为什么我们在代码里看到的表格是这样的。

### 预热：理解 `zip()` 的"配对"

在 Neuron 的前向计算里，我们需要把权重列表 `w` 和输入列表 `x` 逐个相乘：`w[0]*x[0], w[1]*x[1], ...`。

Python 的 `zip()` 就是用来做这种"配对"的：

```python
w = [0.5, -0.3, 0.2]      # 3 个权重
x = [1.0,  2.0, 3.0]      # 3 个输入

# zip(w, x) 把两个列表配成对
for wi, xi in zip(w, x):
    print(f"w={wi}, x={xi}")
# 输出：
# w=0.5, x=1.0
# w=-0.3, x=2.0
# w=0.2, x=3.0
```

这样你就可以一边迭代，一边计算 `wi * xi` 的乘积了。如果你以前只学过 `for i in range(n)` 的写法，这种"直接解包"可能陌生，但用起来很方便——代码一眼就能看出你在配对两个序列。

In [ ]:
# zip() 的最小化验证
w_test = [0.5, -0.3, 0.2]
x_test = [1.0, 2.0, 3.0]
pairs = list(zip(w_test, x_test))
print("zip 的输出（列表形式）:", pairs)
print("逐对计算乘积:", [wi*xi for wi, xi in zip(w_test, x_test)])

### 预热：Value 对象的最小示例——计算图从何而来

初学时，"Value 对象建立计算图"这句话可能很抽象。让我们用最简单的例子看看：

```python
a = Value(2.0)   # 创建一个 Value，存储数据 2.0
b = Value(3.0)   # 创建另一个 Value，存储数据 3.0
c = a + b        # 计算 c = 2.0 + 3.0 = 5.0
# 此时 c 是一个新的 Value，记录"我是由 a 和 b 通过加法生成的"
```

当我们调用 `c.backward()` 时，系统会：
1. 从 `c` 开始（设置 `c.grad = 1.0`）
2. 沿着"谁生成了我"这条链反向走：`c` 是由 `a+b` 生成的
3. 计算局部梯度：`dc/da = 1`, `dc/db = 1`
4. 最终 `a.grad` 和 `b.grad` 都被填上值

**关键**：我们在定义 `c = a + b` 时，`Value.__add__` 的源码记录了"c 是由 a 和 b 生成的"，这就是**计算图**。下面验证一下：

In [ ]:
# 最小示例：1 + 1 = 2，然后 backward()
a = Value(1.0)
b = Value(1.0)
c = a + b        # c.data = 2.0，c._prev 记录了 {a, b}
print(f"c.data = {c.data}  (应为 2.0)")
print(f"c 的子节点：{c._prev}  (应包含 a 和 b)")

# backward() 会沿着计算图反向更新梯度
c.backward()
print(f"反向传播后：a.grad = {a.grad}, b.grad = {b.grad}  (都应为 1.0)")
print("✅ Value 计算图运作正常")

### 图解：为什么 `__truediv__` 不能写成 `Value(other.data ** -1)`？

前面 Value 类的 `__truediv__` 里有一段注释：

```python
def __truediv__(self, other):
    # 必须用 other**-1（调用 __pow__），而非 Value(other.data**-1)。
    # Value(x**-1) 创建孤立节点：不连接到原来的 other，梯度无法回流。
    # other**-1 以 other 为子节点构建 1/other 节点，梯度链路完整。
    ...
```

文字说"孤立节点梯度无法回流"，光看文字可能还是抽象。借着刚才 `c._prev` 的经验，我们把两种写法的计算图画出来对比一下。

**先打个比方**：计算图就像一张"亲子关系树"——每个 Value 出生时，都要在 `_prev` 里登记"我的父母是谁"。`backward()` 沿着这张亲子树，从孩子走向父母，把梯度一路传回去。如果一个节点的 `_prev` 是空的（没有登记任何父母），它就是这张树之外的"孤儿"——梯度传到这里就是终点，不会再往前走了。

**❌ 错误写法：`Value(other.data ** -1)`**

```
   other (data=4.0)                 ← 这是真正参与了后续计算的对象
                                        但它从来没被"登记"为任何节点的父母！

   Value(other.data ** -1)  =  Value(0.25)     ← 凭空造了一个新对象
        _prev = {}                              (空集合，没有任何父节点)
        data  = 0.25   (数值上等于 1/other，但只是"抄了个数字"过来)

   result = self * Value(0.25)
        _prev = {self, Value(0.25)}

   backward() 从 result 出发，能走到 self，也能走到 Value(0.25)，
   但 Value(0.25)._prev 是空的——路走到这里就断了。
   other 这个真正的对象，从头到尾都没出现在任何一条 _prev 边上，
   所以 other.grad 永远停在初始值 0，梯度"回不去"。
```

**✅ 正确写法：`other ** -1`**

```
   other (data=4.0) ──(**-1，调用 __pow__)──→ pow_node (data=0.25)
                                                  _prev = {other}   ← 登记了！

   result = self * pow_node
        _prev = {self, pow_node}

   backward() 从 result 出发 → 走到 pow_node（因为 pow_node ∈ result._prev）
                              → 再走到 other（因为 other ∈ pow_node._prev）
   一条完整的路径，梯度顺着这条路一步步传回 other，
   other.grad 会被正确地累加上一个非零值。
```

**一句话总结**：`other ** -1` 和 `Value(other.data ** -1)` 算出来的**数值**一模一样（都是 0.25），区别只在于前者"记得自己是从 other 算出来的"，后者不记得。反向传播靠的不是数值，而是这张"谁生成了谁"的记录——数值算对了，记录漏掉了，梯度照样传不回去。

下面用代码把两种写法都跑一遍，亲眼看看 `other.grad` 的区别：

In [ ]:
# 亲眼验证："孤立节点" vs "完整节点" 计算图的区别

# ❌ 错误写法：手动算出 1/other，套进一个全新的 Value
other_bad = Value(4.0)
isolated = Value(other_bad.data ** -1)     # data=0.25，但 _prev 是空的
result_bad = Value(2.0) * isolated
result_bad.backward()

print("❌ 错误写法 Value(other.data ** -1)：")
print(f"   isolated._prev = {isolated._prev}   (空集合——孤儿节点)")
print(f"   other_bad.grad = {other_bad.grad}   (卡在 0.0，梯度传不回去！)")

print()

# ✅ 正确写法：other ** -1，pow_node 记住了 other 是自己的父节点
other_good = Value(4.0)
pow_node = other_good ** -1                # data=0.25，_prev={other_good}
result_good = Value(2.0) * pow_node
result_good.backward()

print("✅ 正确写法 other ** -1：")
print(f"   pow_node._prev = {pow_node._prev}   (包含 other_good 本身)")
print(f"   other_good.grad = {other_good.grad}   (正确算出了非零梯度)")

In [ ]:
# 演示：手动搭一个 nin=2 的 Neuron（还没有 class，先把运算写出来）
random.seed(42)
w = [Value(random.uniform(-1, 1)) for _ in range(2)]
b  = Value(0.0)

x  = [Value(1.5), Value(-0.5)]

# 前向：dot + bias + tanh
act = sum((wi * xi for wi, xi in zip(w, x)), Value(0.0)) + b
out = act.tanh()

print("w  =", [round(wi.data, 4) for wi in w])
print("b  =", round(b.data, 4))
print("x  =", [xi.data for xi in x])
print("act=", round(act.data, 4))
print("out=", round(out.data, 4), "  (tanh 值在 (-1, 1) 之间)")

### 补充：backward() 如何传梯度——链式法则的自动化

当你调用 `loss.backward()` 时，系统做了什么？

**初始化**：
```python
loss.grad = 1.0   # 从输出开始，设置 dloss/dloss = 1.0
```

**逆序遍历计算图**：
Value 类在 `__init__` 时记录了"我是由谁生成的"（存在 `_prev` 集合里）。backward() 的核心是：

```python
# 伪代码
for node in reverse_topological_order(graph):
    node._backward()   # 调用这个节点的本地梯度计算函数
```

**本地梯度是什么？**

每个操作（+、×、tanh 等）都有自己的梯度规则。例如：

```python
# 加法：c = a + b
# dc/da = 1, dc/db = 1
def add_backward():
    a.grad += c.grad * 1      # c 对 a 的局部梯度 = 1
    b.grad += c.grad * 1      # c 对 b 的局部梯度 = 1

# 乘法：c = a * b
# dc/da = b, dc/db = a
def mul_backward():
    a.grad += c.grad * b.data    # c 对 a 的局部梯度 = b
    b.grad += c.grad * a.data    # c 对 b 的局部梯度 = a

# tanh：y = tanh(z)
# dy/dz = 1 - tanh(z)^2
def tanh_backward():
    z.grad += y.grad * (1 - y.data**2)
```

**链式法则的自动化**：

当你看到 `a.grad += c.grad * local_gradient` 这样的代码，实际上在计算：

```
da/dx = da/dc × dc/dx
        └─ 上游梯度 ─┘  └局部梯度┘
```

如果 a → c → loss，那么：

```
dloss/da = dloss/dc × dc/da
         = c.grad (已在前一步算好) × local_gradient
```

**具体例子**

```python
a = Value(2.0)
b = Value(3.0)
c = a + b           # c.grad = 1.0（从 loss.backward() 设置）
loss = c * Value(2.0)

loss.backward()
# Step 1：loss 的 backward（乘法）
#   a.grad += loss.grad * b.data  = 1.0 * 2.0 = 2.0
#   c.grad += loss.grad * a.data  = 1.0 * 2.0 = 2.0

# Step 2：c 的 backward（加法，从 loss.grad 往上传）
#   a.grad += c.grad * 1  = 2.0 + 2.0 * 1 = 4.0
#   b.grad += c.grad * 1  = 0.0 + 2.0 * 1 = 2.0

print(f"a.grad = {a.grad}, b.grad = {b.grad}")
# a.grad = 4.0（经过了两层传播）
# b.grad = 2.0（只经过了加法一层）
```

In [ ]:
# 梯度传播的实际演示
a = Value(2.0)
b = Value(3.0)
c = a + b           # c = 5.0
loss = c * Value(2.0)  # loss = 10.0

print(f"前向传播：a={a.data}, b={b.data}, c={c.data}, loss={loss.data}")
print(f"梯度初始化：a.grad={a.grad}, b.grad={b.grad}, c.grad={c.grad}, loss.grad={loss.grad}")

loss.backward()

print(f"\nbackward() 后：")
print(f"  a.grad = {a.grad}  (应为 2.0，因为 dloss/da = dloss/dc × dc/da = 2.0 × 1.0)")
print(f"  b.grad = {b.grad}  (应为 2.0，因为 dloss/db = dloss/dc × dc/db = 2.0 × 1.0)")
print(f"  c.grad = {c.grad}  (应为 2.0，因为 dloss/dc = 2.0（从乘法）)")

# 再看一个复杂的例子：a 通过两条路径影响 loss
print("\n" + "="*50)
print("复杂例子：a 通过多条路径到 loss")
a2 = Value(2.0)
c2 = a2 * a2        # c = a * a = 4
d2 = c2 + a2        # d = c + a = 4 + 2 = 6
loss2 = d2 * Value(3.0)  # loss = d * 3 = 18

print(f"前向：a2={a2.data}, c2={c2.data}, d2={d2.data}, loss2={loss2.data}")
loss2.backward()

print(f"backward() 后：a2.grad = {a2.grad}")
print(f"  预期：dloss/da = 3×(dc/da + dd/da)")
print(f"              = 3×(2×a + 1)")
print(f"              = 3×(2×2 + 1)")
print(f"              = 3×5 = 15")
print(f"  实际值：{a2.grad}  ✅")


### 补充：`sum()` 在 Value 对象上的工作原理

在 Neuron 的计算里，你会看到这行代码：

```python
act = sum((wi * xi for wi, xi in zip(self.w, x)), Value(0.0)) + self.b
```

看起来复杂，但逻辑很简单——让我们逐步拆开：

```python
# 第一步：生成所有乘积（生成器表达式）
products = (wi * xi for wi, xi in zip(self.w, x))  # 没有执行，只是定义

# 第二步：sum() 逐个加起来
result = sum(products, Value(0.0))
# 等价于：
# result = Value(0.0)
# for product in products:
#     result = result + product
```

**为什么第二个参数是 `Value(0.0)`？**

在 Python 的 `sum()` 函数里，第二个参数是**起始值**。比如：

```python
sum([1, 2, 3], 0)      # 0 + 1 + 2 + 3 = 6
sum([1, 2, 3], 10)     # 10 + 1 + 2 + 3 = 16
```

对于 Value 对象也一样：

```python
sum([v1, v2, v3], Value(0.0))   # 从 Value(0.0) 开始，逐个加
                                # = Value(0.0) + v1 + v2 + v3
```

**为什么不能写 `sum(...)`（不带第二参数）？**

如果你写 `sum((wi * xi for ...), 0)`（用普通的 0），前几步会是：

```
0 + (w[0]*x[0])   # 这里 0 是 int，(w[0]*x[0]) 是 Value
```

Python 的 `int.__add__` 根本不知道怎样加 Value 对象，会出错。所以起始值**必须是 Value 对象**，这样 `Value.__add__` 才能接管加法操作，计算图才能建立。

让我们看个例子：

In [ ]:
# sum() 和 Value 对象的例子
v_list = [Value(2.0), Value(3.0), Value(5.0)]

# ✅ 正确方式：用 Value(0.0) 作起始值
result_correct = sum(v_list, Value(0.0))
print(f"sum(Value 列表, Value(0.0)) = {result_correct.data}  (应为 10.0)")

# ❌ 错的方式：用普通 0 作起始值（会出错）
try:
    result_wrong = sum(v_list, 0)
    print(f"sum(Value 列表, 0) = {result_wrong}")  # 这里 0 + Value(2.0) 会出错
except TypeError as e:
    print(f"❌ sum(Value 列表, 0) 失败：{type(e).__name__}: int 不知道怎样加 Value")

# 验证计算图
result_correct.backward()
for i, v in enumerate(v_list):
    print(f"v_list[{i}].grad = {v.grad}  (都应为 1.0，说明梯度传播正常)")

### 补充：tanh vs relu——为什么这课选 tanh？

你可能注意到注释说"原版 micrograd 用 relu，本课用 tanh"。两者区别是什么？

| 激活函数 | 输出范围 | 公式 | 什么情况用 |
|---------|--------|------|----------|
| **tanh** | (-1, 1) | tanh(z) = (e^z - e^{-z}) / (e^z + e^{-z}) | 本课的隐藏层 |
| **relu** | [0, ∞) | relu(z) = max(0, z) | 现代深度学习（更快，更稀疏） |

**为什么本课选 tanh？**

1. **对称性好**：输出中心在 0（-1 到 1），梯度更稳定
2. **便于观察**：输出数值都在 (-1, 1) 之间，数值较小，容易打印和调试
3. **梯度计算简单**：d(tanh)/dz = 1 - tanh(z)^2（平滑，易于理解）

**relu 的优势**：
- **计算快**：relu 就是 `max(0, z)`，没有指数运算
- **稀疏激活**：负数直接变 0，网络自动学到哪些特征不重要
- **深度网络友好**：梯度不会像 tanh 那样"饱和"（当 |z| 很大时，tanh 梯度接近 0）

**现在你有了 Value.relu() 方法**（代码里能看到），如果想试试 relu，只需把代码里的 `.tanh()` 改成 `.relu()` 就行。

让我们快速对比一下梯度：

In [ ]:
import math
import numpy as np

# 对比 tanh 和 relu 的输出和梯度
z_vals = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])

print("z 值       | tanh(z)    | d(tanh)/dz | relu(z)  | d(relu)/dz")
print("-" * 65)
for z in z_vals:
    tanh_z = math.tanh(z)
    d_tanh = 1 - tanh_z**2
    relu_z = max(0, z)
    d_relu = 1.0 if z > 0 else 0.0
    print(f"{z:6.1f}     | {tanh_z:8.4f}   | {d_tanh:8.4f}  | {relu_z:7.1f} | {d_relu:8.1f}")

print("\n观察：")
print("- tanh 的梯度总是正数，范围 [0, 1]，当 |z| 很大时接近 0（梯度消失）")
print("- relu 的梯度要么 0 要么 1（二进制），计算快，负数完全关闭，正数完全开启")

## 2. Layer — 并行的一组 Neuron

`Layer(nin, nout)` 包含 `nout` 个独立的 `Neuron(nin)`，它们接收**同一个输入向量 `x`**，各自独立计算，输出 `nout` 个标量组成的列表：

```
[neuron_0(x), neuron_1(x), ..., neuron_{nout-1}(x)]
```

`nout=1` 时输出列表长度为 1（最终回归层的常见设置）。每个 `Neuron` 的参数相互独立，梯度互不干扰。

### 深入理解：Layer 的"智能返回"——为什么单输出返回 Value？

在 Layer.__call__ 的代码注释里，你会看到一个设计选择：

```python
if len(outs) == 1:
    return outs[0]      # 单输出时返回 Value（标量）
else:
    return outs         # 多输出时返回列表
```

这看起来很方便，但为什么要这样设计？

**问题的核心**：多层 MLP 如何串联？

想象你有三个 Layer：
- Layer1(3, 4) 输出 4 个值
- Layer2(4, 2) 输入应该是 4 个值
- Layer3(2, 1) 输入应该是 2 个值，**输出只有 1 个**

**两种设计方案**：

方案 A（始终返回列表）：
```python
out1 = layer1(x)          # 返回 [v0, v1, v2, v3]
out2 = layer2(out1)       # 返回 [v4, v5]
out3 = layer3(out2)       # 返回 [v6]  <- 虽然只有 1 个，也是列表
loss = out3[0]            # 要取出来还得用 [0]
```
最后用 loss 时还要 `out3[0]`，很烦。

方案 B（智能返回，本课采用）：
```python
out1 = layer1(x)          # 返回 [v0, v1, v2, v3]
out2 = layer2(out1)       # 返回 [v4, v5]
out3 = layer3(out2)       # 自动返回 v6（单值 Value，不是列表）
loss = out3               # 直接用，不需要 [0]
```

**代价**：代码里需要在 MLP.__call__ 里"拦截"一下，确保即使最后一层是单值，也统一包装成列表返回给外界（下节课 training loop 会需要）。

这就是**接口设计**：在适当的层面牺牲一点一致性，换取使用时的便利。

In [ ]:
# 演示：手动搭 nout=3 的 Layer（nin=2 → 输出 3 个标量）
random.seed(0)
nin, nout = 2, 3
# 3 个 Neuron，每个有 2 个权重
neurons_demo = [
    [Value(random.uniform(-1, 1)) for _ in range(nin)]
    for _ in range(nout)
]
biases_demo = [Value(0.0) for _ in range(nout)]

x_demo = [Value(1.0), Value(-1.0)]

outputs_demo = []
for ws, b in zip(neurons_demo, biases_demo):
    act = sum((w * xi for w, xi in zip(ws, x_demo)), Value(0.0)) + b
    outputs_demo.append(act.tanh())

print(f"Layer(nin=2, nout=3) 输出 {len(outputs_demo)} 个 Value:")
for i, o in enumerate(outputs_demo):
    print(f"  out[{i}] = {o.data:.4f}")

### 深入理解：如果中间层的 nout=1，"智能返回"会不会出问题？

有个很尖锐的问题值得深挖：**Layer 有时候返回一个 Value，有时候返回一个列表——那两层首尾相连的时候，前一层"随手"返回的东西，后一层能直接接住吗？**

先看看 MLP 通常长什么样：`MLP(3, [4, 4, 1])`——前两层的 `nout` 都是 4（≥2），只有**最后一层**的 `nout=1`。这不是巧合。我们来看看，如果**中间**也出现一个 `nout=1` 的层，会发生什么。

**回忆 Neuron 的前向代码**：
```python
act = sum((wi * xi for wi, xi in zip(self.w, x)), Value(0.0)) + self.b
```
这里的 `x` 必须是**可迭代的**（比如列表），因为 `zip(self.w, x)` 需要把 `self.w` 和 `x` 逐个配对。如果 `x` 是裸的一个 `Value` 对象（不是列表），`zip()` 根本不知道怎么"配对"——它会直接报错，因为 `Value` 对象没有实现 `__iter__`，不能被迭代。

**具体会怎样出错？**用一个"瓶颈层"（中间层 nout=1）实际跑一遍就知道了：

In [ ]:
# 演示：中间层 nout=1 时，"智能返回"会不会撞车？
# （这里手写两个演示用的 mini class，不影响你后面自己实现的 Neuron/Layer）

class _DemoNeuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), Value(0.0)) + self.b
        return act.tanh()

class _DemoLayer:
    def __init__(self, nin, nout):
        self.neurons = [_DemoNeuron(nin) for _ in range(nout)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs   # 就是"智能返回"

random.seed(7)
layer_a = _DemoLayer(3, 1)   # 中间层，nout=1（瓶颈层）——不是网络的最后一层！
layer_b = _DemoLayer(1, 2)   # 下一层，nin=1，期望接收"1 个元素的列表"

x_in = [Value(1.0), Value(2.0), Value(3.0)]
out_a = layer_a(x_in)
print(f"layer_a(nout=1) 的输出类型：{type(out_a).__name__}")
print("  ← 因为智能返回规则，nout=1 时直接返回裸 Value，不是 [Value]！")

try:
    out_b = layer_b(out_a)   # layer_b 内部的 Neuron 会尝试 zip(self.w, out_a)
    print("layer_b 输出：", out_b)
except TypeError as e:
    print(f"\n❌ 报错了！TypeError: {e}")
    print("原因：out_a 是裸 Value，不可迭代（not iterable）；")
    print("      zip(self.w, out_a) 没办法把 self.w 的元素和 out_a 逐个配对。")

**这说明了什么？**

"智能返回"这个设计是有代价的——它默认**只有网络的最后一层**会把结果直接交给外部使用者（人类或 loss 函数），中间层的输出永远只是"喂给下一层"的内部数据，**不应该**出现 nout=1 的情况，否则下一层的 `zip()` 就会像上面这样直接报错。

**这门课的例子如何避开这个坑？**看 `MLP(3, [4, 4, 1])`——宽度序列是 `[4, 4, 1]`，只有**最后一个数字**是 1，前面的隐藏层宽度都 ≥ 2。这是一种**约定**，不是代码自动保证的：如果你自己设计网络结构时写出类似 `MLP(3, [4, 1, 4])`（中间藏了一个瓶颈层），前向传播会在中间层和下一层的衔接处报错，就像上面演示的那样。

**MLP 内部真正统一处理类型差异的地方，只有一处**：`MLP.__call__` 在最后把**整个网络的最终输出**统一包装成列表再返回给外部调用者（下一个小节会详细展开这一点）。层与层之间的"传递"（也就是 `x = layer(x)` 那一步）没有额外的保护——这是这个从零实现版本的一个已知限制。工业界的框架（比如 PyTorch）用统一的张量（Tensor）格式来避开这个问题：无论"批大小"是 1 还是 100，输出的形状规则永远一致，不会出现"有时候是标量，有时候是列表"的歧义。

## 3. MLP — 多层串联

`MLP(nin, nouts)` 把 `len(nouts)` 个 `Layer` 首尾相连：第 0 层接收原始输入（维度 `nin`），第 `k` 层接收第 `k-1` 层的输出。每层的 `nin` 恰好是上一层的 `nout`：

```
Layer(nin,       nouts[0])   → 输出 nouts[0] 维
Layer(nouts[0],  nouts[1])   → 输出 nouts[1] 维
...
Layer(nouts[-2], nouts[-1])  → 最终输出
```

最后一层通常关闭非线性（`nonlin=False`）。`parameters()` 遍历所有层的所有 `Neuron`，把全部 `w` 和 `b` 拍成一个平坦列表——训练时对这个列表里每个 `Value` 执行 `p.data -= lr * p.grad` 就完成一步梯度下降（gradient descent，GD）。

### 深入理解：为什么需要 `parameters()` 方法？

MLP 有嵌套结构：MLP → Layer → Neuron → Value。要访问所有参数，你需要一路往下钻：

```python
# ❌ 不方便的写法
for layer in mlp.layers:
    for neuron in layer.neurons:
        for param in neuron.parameters():
            param.data -= lr * param.grad
```

还要三重循环！想象有 100 层的网络，代码会很丑。

**解决方案：`parameters()` 方法，一层层往下拍平**

```python
# ✅ 简洁的写法
for param in mlp.parameters():
    param.data -= lr * param.grad
```

`mlp.parameters()` 内部调用 `layer.parameters()`，每个 `layer.parameters()` 调用所有 `neuron.parameters()`，最后返回一个**平坦列表**：

```
mlp.parameters() 
  → [所有 Layer0 的参数, 所有 Layer1 的参数, ..., 所有 LayerN 的参数]
  → [所有 Neuron 的 w 和 b，拍成一个列表]
```

**为什么拍平？**

1. **训练时方便**：一个简单的 for 循环就能遍历所有 41 个参数
2. **优化器友好**：SGD、Adam 等优化器都期望一个参数列表，而不是嵌套结构
3. **和框架一致**：PyTorch 的 `model.parameters()` 也返回生成器，遍历所有参数

下节课 L58 的训练循环会用到这个方法。

### 深入理解：为什么最后一层要关闭非线性？

这是初学者常见的困惑。隐藏层都过 tanh（压到 -1 到 1），为什么最后一层不能这样做？

**场景 1：回归任务（预测房价）**

想象你在预测房价。房价可以是任意正数：$50,000, $500,000, $5,000,000...

- 如果最后一层过 tanh，输出被压到 (-1, 1)，比如 tanh(z) = 0.95
- 你怎样把 0.95 转换回真实的房价？不行！因为信息丢失了
- 最后一层**必须是线性的**（不过激活函数），这样输出 z 可以是任意实数

```python
# ❌ 错的方式：最后一层过 tanh
pred = tanh(w[50]*h[50] + b)  # 输出被压到 (-1, 1)，无法预测大的房价

# ✅ 正确方式：最后一层线性
pred = w[50]*h[50] + b         # 输出可以是任意实数，比如 250000.5
```

**场景 2：分类任务（是否是猫）**

二分类：输出一个概率（0 到 1）或 logit（任意实数）。

- 最后一层输出 logit（无限制的实数）后，再过 sigmoid 或 softmax 转换为概率
- **最后一层仍然是线性的**，激活函数放在外面

**隐藏层为什么要非线性？**

隐藏层过激活函数（tanh/relu）有不同目的：
- 引入**非线性变换**，让网络能学习复杂的关系（不光是线性组合）
- 压缩数据范围，稳定训练

但**最后一层不需要这个**，它的工作是直接输出原始的线性组合。

In [ ]:
# 演示：用嵌套列表模拟 MLP(3, [4, 4, 1]) 的层结构
# （先打印结构，后面才实现真正的 class）
layer_shapes = [(3, 4), (4, 4), (4, 1)]
total_params = sum(nin * nout + nout for nin, nout in layer_shapes)
print("MLP(3, [4, 4, 1]) 参数统计：")
for i, (n_in, n_out) in enumerate(layer_shapes):
    w_count = n_in * n_out
    b_count = n_out
    print(f"  Layer {i}: {n_in}×{n_out} 权重 + {n_out} 偏置 = {w_count + b_count}")
print(f"  总计：{total_params} 个可训练参数")

In [ ]:
class Neuron:
    def __init__(self, nin, nonlin=True):
        # ✏️ TODO: 初始化 self.w（nin 个 Value，均匀随机 [-1, 1)）和 self.b（Value(0.0)）
        # ✏️ TODO: 保存 self.nonlin
        raise NotImplementedError("TODO: 初始化 self.w（nin 个随机 Value）、self.b（Value(0.0)）并保存 self.nonlin")

    def __call__(self, x):
        # ✏️ TODO: 计算 act = sum(wi * xi for ...) + self.b
        # ✏️ TODO: 若 self.nonlin 为 True，返回 act.tanh()，否则返回 act
        raise NotImplementedError("TODO: 计算加权求和 act，若 nonlin 则返回 act.tanh()，否则返回 act")

    def parameters(self):
        # ✏️ TODO: 返回 self.w + [self.b]
        raise NotImplementedError("TODO: 返回 self.w + [self.b] 的参数列表")

    def __repr__(self):
        kind = "Tanh" if self.nonlin else "Linear"
        return f"{kind}Neuron({len(self.w)})"


In [ ]:
# ── Neuron 单元验证 ──
try:
    import random; random.seed(0)
    n = Neuron(3)
    assert len(n.w) == 3, f'Neuron(3) 应有 3 个权重，实际 {len(n.w)}'
    assert isinstance(n.b, Value), 'b 应为 Value'
    out = n([Value(1.0), Value(0.5), Value(-1.0)])
    assert isinstance(out, Value), '__call__ 应返回 Value'
    params = n.parameters()
    assert len(params) == 4, f'Neuron(3) 应有 4 个参数，实际 {len(params)}'
    print('✅ Neuron 通过：w×3 + b，__call__ 返回 Value，parameters() 长度=4')
except (NotImplementedError, TypeError):
    print('⬜ Neuron 未实现')
except NameError:
    print('⬜ 请先运行定义 Value 的 cell（来自 L55/L56）')


In [ ]:
class Layer:
    def __init__(self, nin, nout, **kwargs):
        # ✏️ TODO: self.neurons = nout 个 Neuron(nin, **kwargs)
        raise NotImplementedError("TODO: 创建 self.neurons，包含 nout 个 Neuron(nin, **kwargs)")

    def __call__(self, x):
        # ✏️ TODO: outs = [n(x) for n in self.neurons]
        # ✏️ TODO: 若 len(outs)==1 返回 outs[0]，否则返回 outs
        raise NotImplementedError("TODO: 前向传播：列表推导得 outs，len==1 时返回标量否则返回列表")

    def parameters(self):
        # ✏️ TODO: 返回所有 neuron 的 parameters() 拍平后的列表
        raise NotImplementedError("TODO: 展平所有 neuron.parameters() 并返回单一列表")

    def __repr__(self):
        return f"Layer([{', '.join(str(n) for n in self.neurons)}])"


In [ ]:
# ── Layer 单元验证 ──
try:
    import random; random.seed(1)
    layer = Layer(2, 3)
    assert len(layer.neurons) == 3, f'Layer(2,3) 应有 3 个 Neuron，实际 {len(layer.neurons)}'
    outs = layer([Value(0.5), Value(-0.5)])
    assert isinstance(outs, list) and len(outs) == 3, 'Layer(2,3) 输出应为长度 3 的列表'
    assert len(layer.parameters()) == 9, f'Layer(2,3) 应有 9 个参数，实际 {len(layer.parameters())}'
    print('✅ Layer 通过：3 neurons，输出列表长=3，parameters() 长度=9')
    # 单输出层应返回 Value 而非列表
    single = Layer(2, 1)([Value(0.5), Value(-0.5)])
    assert isinstance(single, Value), 'Layer(2,1) 应返回单个 Value'
    print('✅ Layer(2,1) 单输出时返回 Value（非列表）')
except (NotImplementedError, TypeError):
    print('⬜ Layer 未实现')
except NameError:
    print('⬜ 请先实现 Neuron')


### 深入理解：为什么 MLP.__call__ 最后要"统一包装为列表"？

马上要实现的 `MLP.__call__` TODO 里有一句关键的注释：

> 统一返回列表（若结果不是 list 则 `[x]`）

这句话什么意思？为什么要这样做？先打个比方。

**类比一下**：快递员送货，不管你这次买了 1 件还是 3 件商品，快递员永远给你一个包裹（哪怕里面只装了 1 件东西）。你收货时不用先问"这次是一件东西直接塞我手上，还是装箱给我？"——永远打开同一个包裹就行。如果快递员心情好就直接把东西塞你手上，心情不好就装箱，你每次收货前都得先猜一下"这次是哪种"，非常容易出错。

**回到 MLP**：还记得 `Layer` 的"智能返回"吗——`nout=1` 时返回裸 `Value`，否则返回列表。`MLP` 的最后一层，如果是回归任务，往往 `nout=1`（比如预测一个房价数字），这一层单独调用后会返回一个裸 `Value`，不是列表。

如果 `MLP.__call__` 不做任何处理，直接把最后一层的结果原样返回，那么：
- `MLP(3, [4, 4, 1])(x)` → 返回一个裸 `Value`
- `MLP(3, [4, 4, 2])(x)` → 返回一个 `[Value, Value]` 列表

**这会带来什么麻烦？**下一课 `L58` 的训练循环，需要把"模型输出"和"训练目标（标签）"逐个配对算 loss，代码大概长这样：

```python
# L58 训练循环的核心一步（预告）
for yp, yt in zip(ypred, ygt):     # 假设 ypred 和 ygt 都是列表
    loss += (yp - yt) ** 2
```

如果 `ypred` 有时候是列表、有时候是裸 `Value`，这个 `zip(ypred, ygt)` 有时候能跑，有时候会报错（就像上一个小节我们亲眼看到的 `TypeError` 一样）。写训练代码的人就必须每次都先判断"这次是 Value 还是列表"，非常容易漏掉、出 bug。

**解决办法**：让 `MLP.__call__` 在**返回给外部调用者之前**，做最后一次"统一包装"——不管最后一层给出的是裸 `Value` 还是列表，出了 `MLP.__call__` 的门，永远是一个列表（哪怕只有 1 个元素）。这样 `L58` 的训练代码可以放心地写 `zip(ypred, ygt)`，不用对"1 个输出" 和"多个输出"两种情况分别处理——这也是为什么这一步必须做，而不是"锦上添花"的选项。

**具体怎么实现？**逻辑很直白——判断结果是不是 list，不是就用 `[x]` 包一层：

```python
def unify_output(raw_output):
    return raw_output if isinstance(raw_output, list) else [raw_output]
```

跑一下看看两种情况分别会发生什么，以及它如何让下一课的训练代码免受类型困扰：

In [ ]:
# 演示：MLP.__call__ 最后"统一包装为列表"的逻辑

def unify_output(raw_output):
    # 这正是 MLP.__call__ 里 "统一返回列表" TODO 要实现的逻辑
    return raw_output if isinstance(raw_output, list) else [raw_output]

# 情形 1：最后一层 nout=1（回归任务），Layer 返回裸 Value
single_val = Value(3.14)
print("情形1（回归，nout=1）")
print("  统一包装前：", single_val, " 类型：", type(single_val).__name__)
print("  统一包装后：", unify_output(single_val))

print()

# 情形 2：最后一层 nout>=2（多输出任务），Layer 返回列表
multi_val = [Value(1.0), Value(2.0)]
print("情形2（多输出，nout=2）")
print("  统一包装前：", multi_val)
print("  统一包装后：", unify_output(multi_val))

print()

# 预告 L58 训练循环：有了统一的列表，zip() 就能放心用
ygt = [1.0]                          # 1 个训练样本，1 个目标值
ypred = unify_output(single_val)     # 统一之后，不管原来是不是列表，现在肯定是列表
loss = sum((yp - Value(yt))**2 for yp, yt in zip(ypred, ygt))
print(f"预告 L58：loss = {loss.data:.4f}  (zip(ypred, ygt) 不用担心类型问题)")

In [ ]:
class MLP:
    def __init__(self, nin, nouts):
        # ✏️ TODO: sz = [nin] + nouts
        # ✏️ TODO: self.layers = Layer(sz[i], sz[i+1], nonlin=...) 列表
        #          最后一层 nonlin=False，其余 True
        raise NotImplementedError("TODO: 构建 sz 列表，创建 self.layers（最后一层 nonlin=False，其余 True）")

    def __call__(self, x):
        # ✏️ TODO: 依次把 x 喂入每一层（x = layer(x)）
        # ✏️ TODO: 统一返回列表（若结果不是 list 则 [x]）
        raise NotImplementedError("TODO: 串联各层前向传播，最终结果统一包装为列表返回")

    def parameters(self):
        # ✏️ TODO: 返回所有层所有参数的平坦列表
        raise NotImplementedError("TODO: 展平所有 layer.parameters() 并返回单一列表")

    def __repr__(self):
        return f"MLP([{', '.join(str(l) for l in self.layers)}])"


In [ ]:
# ── MLP 综合验证（未实现时打印提示而非崩溃） ──
try:
    random.seed(0)

    # 基础结构检查
    m = MLP(2, [3, 1])
    out = m([Value(0.5), Value(-0.5)])
    assert isinstance(out, list) and len(out) == 1, "MLP 输出应为长度 1 的列表"
    assert isinstance(out[0], Value), "MLP 输出元素应为 Value"
    print("✅ MLP(2,[3,1]) 前向传播通过，out[0].data =", round(out[0].data, 4))

    # 参数数量检查
    m2 = MLP(3, [4, 4, 1])
    params = m2.parameters()
    assert len(params) == 41, f"参数数量应为 41，实际 {len(params)}"
    print("✅ MLP(3,[4,4,1]) 参数数量正确：", len(params))

    # 反向传播检查
    out2 = m2([Value(1.0), Value(2.0), Value(3.0)])
    loss = out2[0]
    loss.backward()
    grads = [p.grad for p in params]
    assert any(g != 0.0 for g in grads), "backward() 后参数梯度应不全为 0"
    print("✅ backward() 梯度传播正常，首个参数梯度 =", round(params[0].grad, 6))
except (NotImplementedError, TypeError):
    print('⬜ TODO：请先完成 Neuron / Layer / MLP 三个 class，再运行本格验证。')

## 参数实验：验证 MLP(3,[4,4,1]) 的参数总数

手动推算三层的参数数量，再用 `len(m.parameters())` 核对：

| 层 | nin | nout | 权重数 `nin × nout` | 偏置（bias，b）数 `nout` | 小计 |
|----|-----|------|---------------------|---------------|------|
| 0  |  3  |  4   | 12                  | 4             | **16** |
| 1  |  4  |  4   | 16                  | 4             | **20** |
| 2  |  4  |  1   |  4                  | 1             | **5**  |
| 合计 | — | — | — | — | **41** |

预期现象：`len(m.parameters()) == 41`，且 `Neuron.__repr__` 最后一层显示 `LinearNeuron`（无非线性）。

In [ ]:
# ── 参数总数核对（未实现时打印提示而非崩溃） ──
try:
    random.seed(99)
    m3 = MLP(3, [4, 4, 1])
    print("网络结构：", m3)
    print()

    for i, layer in enumerate(m3.layers):
        lp = layer.parameters()
        print(f"Layer {i}: {len(lp):3d} 个参数  |  neurons: {layer.neurons}")

    total = len(m3.parameters())
    print(f"\n总参数：{total}  （预期 41）")

    # 检查最后一层是否关闭了非线性
    assert not m3.layers[-1].neurons[0].nonlin, "最后一层 Neuron 应 nonlin=False（线性输出）"
    print("✅ 最后一层为线性（nonlin=False）")
except (NotImplementedError, TypeError):
    print('⬜ TODO：请先完成 Neuron / Layer / MLP 三个 class，再运行本格核对参数数。')

## 本课收束

本节实现了 `Neuron`（单个加权求和+激活）、`Layer`（并行 `nout` 个 Neuron）、`MLP`（多层串联），`parameters()` 返回全部 41 个可训练 `Value` 的平坦列表，让梯度下降一个个更新它们。这套结构和 PyTorch `nn.Module` 完全同构，也是 Aurora ML 引擎（`src/aurora/ml/`，计划中模块）里网络主体的原型。下一节 `L58_training_loop.ipynb` 将用这个 MLP 拟合 `make_moons` 双月牙数据集（sklearn 缺失时用 NumPy 从零生成），以 hinge loss 为目标，串起「前向 → 算 loss → backward() → 更新参数 → 清零梯度」的完整训练循环，验收标准是分类准确率 > 85%。

## ✏️ 白板挑战：MLP 结构手推（目标 10 分钟）

盖上屏幕，纸上完成：

**问 1**：`MLP(2, [3, 1])` 共有多少个参数？（列出每层的 weight + bias 数）  
（Layer1: 3×(2+1)=9，Layer2: 1×(3+1)=4，共 13 个）

**问 2**：`Layer(4, 5)` 的 `forward(x)` 输出是什么形状？输入 x 是什么形状？  
（输入：4个标量的列表，输出：5个 Value 的列表）

**问 3**：为什么最后一层 `MLP` 通常 `nonlin=False`？  
（最后一层是回归输出或 logit，不需要 relu 截断负值）

**问 4**：`Neuron.parameters()` 返回什么？为什么需要这个方法？  
（返回 w + [b]，训练时需要遍历所有参数做 `p.grad = 0.0` 清零）

**问 5**：若 `MLP(3, [4,4,1])` 输入 `[1.0, 2.0, 3.0]`，前向传播后有多少个 `Value` 节点（大约）？  
（每个 Neuron 约 nin+1 次加法+1次激活，第1层4个Neuron≈20节点，第2层4个≈20，第3层1个≈5，合计约45个节点）

推导完成后运行下方格验证。

## 深入讲解：计算图中的节点数（白板挑战问5的详细推导）

"计算图有多少个节点"这个问题看似简单，但需要理解两件事：

**什么是一个节点？**

计算图中，**每一次运算都是一个节点**。比如：

```python
a = Value(1.0)
b = Value(2.0)
c = a * b      # 一个节点：乘法
d = c + a      # 一个节点：加法
```

这样的计算图有 4 个节点（a, b, c, d）和两次操作（乘法、加法）。

**一个 Neuron 前向传播的细节拆解**

假设 `Neuron(3)` 接收输入 `x = [1.0, 2.0, 3.0]`，权重 `w = [0.5, -0.3, 0.2]`，偏置 `b = 0.1`：

```python
# 计算加权和（逐步推导）
z1 = w[0] * x[0]      # 节点1：乘法
z2 = w[1] * x[1]      # 节点2：乘法
z3 = w[2] * x[2]      # 节点3：乘法
z4 = z1 + z2          # 节点4：加法（前两项的和）
z5 = z4 + z3          # 节点5：加法（+ 第三项）
z6 = z5 + b           # 节点6：加法（+ 偏置）
y = z6.tanh()         # 节点7：激活函数（tanh）
```

**计数**：Neuron(3) 产生 7 个 Value 节点（3 个乘法 + 3 个加法 + 1 个 tanh）

更一般地：
- 乘法：nin 个（每个权重乘一次输入）
- 加法：nin 个（逐步累加）
- 激活：1 个（tanh）
- **小计**：2×nin + 1 个新节点

**MLP(3, [4,4,1]) 的完整计数**

| 层 | 神经元数 | 每个产生的节点 | 该层总节点 |
|----|--------|-------------|---------|
| L0 | 4 | 2×3+1=7 | 4×7 = 28 |
| L1 | 4 | 2×4+1=9 | 4×9 = 36 |
| L2 | 1 | 2×4+1=9 | 1×9 = 9 |
| **合计** | | | **28+36+9 = 73** |

等等，这和教材说的"约 45 个"不一样？

**原因**：教材的估算方式更简化。它用"每个 Neuron 约 nin+1 次加法+1 次激活"来近似，这是**不精确的简化估算**。如果你要精确计数，应该用上面的方法。

要点是：**计算图节点数随着 nin（输入维度）和网络宽度快速增长**。这就是为什么 backward() 虽然梯度正确，但大网络时计算会很昂贵。

（深度学习框架用的技巧就是动态图、计算图优化，避免存储和遍历所有节点。）

In [ ]:
# 实际计数：计算图有多少个节点

# 手工计数一个 Neuron(3) 的计算图
def count_nodes_manual():
    # 输入（3 个，不算新节点，是输入层）
    w = [Value(0.5), Value(-0.3), Value(0.2)]
    x = [Value(1.0), Value(2.0), Value(3.0)]
    b = Value(0.1)
    
    # 逐步计算
    z1 = w[0] * x[0]      # 1 个乘法节点
    z2 = w[1] * x[1]      # 1 个乘法节点
    z3 = w[2] * x[2]      # 1 个乘法节点
    z4 = z1 + z2          # 1 个加法节点
    z5 = z4 + z3          # 1 个加法节点
    z6 = z5 + b           # 1 个加法节点
    y = z6.tanh()         # 1 个 tanh 节点
    
    # 计数新生成的 Value 节点
    new_nodes = [z1, z2, z3, z4, z5, z6, y]
    print(f"Neuron(3) 新产生的节点数：{len(new_nodes)}")
    print(f"  乘法：3 个，加法：3 个，tanh：1 个 → 总计 7 个")
    
    return y

output = count_nodes_manual()
print(f"输出值：{output.data:.4f}")

# 一般公式
print("\n一般规律：")
for nin in [2, 3, 4, 5]:
    nodes_per_neuron = 2 * nin + 1
    print(f"  Neuron({nin})：{nodes_per_neuron} 个新节点")

In [ ]:
# ✏️ 对答案格
import random

# 问1：MLP(2,[3,1]) 参数数 = 13
try:
    random.seed(0)
    m_check = MLP(2, [3, 1])
    params = m_check.parameters()
    assert len(params) == 13, f"参数数={len(params)}，期望13"
    print(f"Q1 ✅  MLP(2,[3,1]) 参数数={len(params)} (Layer1:9 + Layer2:4 = 13)")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  MLP(2,[3,1])参数数: Layer1=3×(2+1)=9, Layer2=1×(3+1)=4, 共13（待实现后验证）")

# 问2：Layer(4,5)输出5个Value
try:
    random.seed(1)
    layer_check = Layer(4, 5)
    x_in = [Value(float(i)) for i in range(4)]
    out_check = layer_check(x_in)
    assert len(out_check) == 5, f"输出长度={len(out_check)}"
    print(f"Q2 ✅  Layer(4,5): 输入4个Value → 输出{len(out_check)}个Value")
except (NotImplementedError, TypeError):
    print(f"Q2 ✅  Layer(4,5): 输入列表长4，输出列表长5（待实现后验证）")

# 问3：最后一层nonlin=False（概念验证）
try:
    random.seed(2)
    mlp_check = MLP(2, [3, 1])
    last_layer = mlp_check.layers[-1]
    assert all(not n.nonlin for n in last_layer.neurons), "最后层应nonlin=False"
    print(f"Q3 ✅  MLP最后层所有Neuron.nonlin=False（回归输出不截断）")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  最后层nonlin=False：回归logit不需relu截断（待实现后验证）")

# 问4：Neuron.parameters()长度
try:
    random.seed(3)
    n_check = Neuron(5)
    assert len(n_check.parameters()) == 6, f"参数数={len(n_check.parameters())}"
    print(f"Q4 ✅  Neuron(5).parameters()={len(n_check.parameters())}个（5个w + 1个b）")
except (NotImplementedError, TypeError):
    print(f"Q4 ✅  Neuron(nin).parameters() = [w0,...,w_(nin-1), b]，共nin+1个（待实现后验证）")

# 问5：参数总数 MLP(3,[4,4,1])
try:
    random.seed(4)
    m3_check = MLP(3, [4, 4, 1])
    n_params = len(m3_check.parameters())
    # L1: 4×(3+1)=16, L2: 4×(4+1)=20, L3: 1×(4+1)=5 → 41
    assert n_params == 41, f"参数数={n_params}"
    print(f"Q5 ✅  MLP(3,[4,4,1]) 参数数={n_params} (L1:16 + L2:20 + L3:5 = 41)")
except (NotImplementedError, TypeError):
    print(f"Q5 ✅  MLP(3,[4,4,1]): L1=4×4=16, L2=4×5=20, L3=1×5=5, 共41（待实现后验证）")
print("\n🎉 MLP 结构白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l57_review = {
    "neuron_class_impl":       None,  # Neuron.__init__/__call__/parameters 正确实现？True/False
    "layer_class_impl":        None,  # Layer.__init__/__call__/parameters 正确实现？True/False
    "mlp_class_impl":          None,  # MLP.__init__/__call__/parameters 正确实现？True/False
    "param_count_understood":  None,  # 能手算 MLP 任意结构的参数总数？True/False
    "whiteboard_passed":       None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l57_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l57_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L57 全部通关！进入 L58：训练循环')

---

→ **下一课**　[L58 · 训练循环](L58_training_loop.ipynb)

> 下节课将学习 **训练循环**：loss 计算、optimizer.step、收敛曲线，拟合 make_moons 数据集。